In [1]:
import os
#os.environ["OPENAI_API_KEY"]=""
os.environ["OLLAMA_HOST"]="http://10.103.12.74:11434"

In [2]:
from __future__ import annotations

import json
import os
import re
import subprocess
import tempfile
from dataclasses import dataclass, asdict
from typing import Any, Dict, Optional

from phi.agent import Agent
from phi.model.ollama import Ollama


# ----------------------------
# Data structures
# ----------------------------

@dataclass
class RunResult:
    ok: bool
    exit_code: int
    stdout: str
    stderr: str
    wall_time_ms: int


@dataclass
class EvaluationResult:
    score_0_to_10: int
    summary: str
    issues: list[str]
    improvements: list[str]


# ----------------------------
# Tool 2: Run code safely-ish
# ----------------------------

def run_python_code(code: str, timeout_s: int = 10) -> Dict[str, Any]:
    """
    Runs python code in a temp file with a timeout.
    Captures stdout/stderr and returns a dict RunResult-like payload.

    NOTE: This is not a hardened sandbox. For untrusted code, use real sandboxing
    (containers, seccomp, firejail, etc).
    """
    import time

    start = time.time()
    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, "main.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(code)

        try:
            proc = subprocess.run(
                ["python", path],
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )
            end = time.time()
            result = RunResult(
                ok=(proc.returncode == 0),
                exit_code=proc.returncode,
                stdout=proc.stdout or "",
                stderr=proc.stderr or "",
                wall_time_ms=int((end - start) * 1000),
            )
            return asdict(result)

        except subprocess.TimeoutExpired as e:
            end = time.time()
            result = RunResult(
                ok=False,
                exit_code=124,
                stdout=e.stdout or "",
                stderr=(e.stderr or "") + "\nTIMEOUT",
                wall_time_ms=int((end - start) * 1000),
            )
            return asdict(result)


# ----------------------------
# Ollama model helper
# ----------------------------

def make_ollama_model(
    model_id: str = "gemma3:1b",
    host: Optional[str] = None,
    timeout_s: Optional[float] = None,
) -> Ollama:
    """
    Creates a Phidata Ollama model instance.

    - model_id: Ollama model tag, e.g. "gemma3:1b"
    - host: optionally set to something like "http://localhost:11434"
            (or via env var OLLAMA_HOST)
    - timeout_s: optional request timeout
    """
    host = host or os.getenv("OLLAMA_HOST")  # e.g. http://localhost:11434
    return Ollama(
        id=model_id,
        host=host,
        timeout=timeout_s,
    )


# ----------------------------
# Tool 1: Code generator agent
# ----------------------------

def make_code_writer(model: Optional[Any] = None) -> Agent:
    return Agent(
        name="CodeWriter",
        model=model or make_ollama_model(model_id="gemma3:1b"),
        description="You write correct, minimal Python code for the given task.",
        instructions=[
            "Return ONLY raw, executable Python source code.",
            "Do NOT wrap the code in objects, variables, JSON, or metadata (e.g., no content=..., no RunResponse, no Message objects).",
            "Do NOT include markdown fences, explanations, comments about the response format, or any extra text.",
            "Output must start directly with valid Python syntax (e.g., import, def, class, or if __name__ == '__main__':).",
            "Follow best coding practices (clear naming, type hints where appropriate, docstrings when useful).",
            "Prefer the Python standard library unless third-party libraries are explicitly required.",
            "Include a main guard when appropriate: if __name__ == '__main__':",
            "Ensure the code is directly runnable as a standalone .py file.",
        ],
        markdown=False,
    )


def generate_python_code1(task: str, writer: Agent) -> str:
    code = writer.run(task, stream=False)
    code = re.sub(r"^```(?:python)?\s*", "", str(code).strip(), flags=re.IGNORECASE)
    code = re.sub(r"\s*```$", "", code.strip())
    return code.strip()


def generate_python_code(task: str, writer: Agent) -> str:
    resp = writer.run(task, stream=False)

    # 1) If the framework returned a structured object with .content, prefer that.
    try:
        content = getattr(resp, "content", None)
        if isinstance(content, str) and content.strip():
            text = content
        else:
            text = str(resp)
    except Exception:
        text = str(resp)

    text = text.strip()

    def _strip_fences(s: str) -> str:
        s = re.sub(r"^\s*```(?:python)?\s*", "", s, flags=re.IGNORECASE)
        s = re.sub(r"\s*```\s*$", "", s)
        return s.strip()

    def _unescape_python_string_literal(s: str) -> str:
        try:
            return bytes(s, "utf-8").decode("unicode_escape")
        except Exception:
            return s

    m = re.search(
        r"Message\(\s*role\s*=\s*['\"]assistant['\"].*?content\s*=\s*(?P<q>['\"])(?P<body>.*?)(?P=q)\s*[,\)]",
        text,
        flags=re.DOTALL,
    )
    if m:
        extracted = m.group("body")
        extracted = _unescape_python_string_literal(extracted)
        extracted = _strip_fences(extracted)
        if extracted:
            return extracted

    m = re.search(
        r"\bcontent\s*=\s*(?P<q>['\"])(?P<body>.*?)(?P=q)\s+\bcontent_type\b",
        text,
        flags=re.DOTALL,
    )
    if m:
        extracted = m.group("body")
        extracted = _unescape_python_string_literal(extracted)
        extracted = _strip_fences(extracted)
        if extracted:
            return extracted

    text = _strip_fences(text)

    if text.startswith("content="):
        m = re.search(
            r"^content\s*=\s*(?P<q>['\"])(?P<body>.*?)(?P=q)\s*$",
            text,
            flags=re.DOTALL,
        )
        if m:
            extracted = _unescape_python_string_literal(m.group("body"))
            extracted = _strip_fences(extracted)
            if extracted:
                return extracted

    return text


# ----------------------------
# Agent: Code reviewer
# ----------------------------

def make_code_reviewer(model: Optional[Any] = None) -> Agent:
    return Agent(
        name="CodeReviewer",
        model=model or make_ollama_model(model_id="gemma3:1b"),
        description="You review Python code using pylint-style feedback.",
        instructions=[
            "You are a strict Python code reviewer.",
            "Review the provided Python code.",
            "Explain issues clearly and suggest improvements.",
            "Focus on correctness, readability, typing, structure, and best practices.",
            "Do NOT rewrite the entire code unless necessary.",
            "provde only the score of the analysis between 1 and 10, 10 being the best score of the code",
        ],
        markdown=False,
    )


# ----------------------------
# Tool 3: Static code analysis with pylint
# ----------------------------

def run_pylint_analysis1(code: str, timeout_s: int = 20) -> Dict[str, Any]:
    import time

    start = time.time()

    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, "main.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(code)

        try:
            proc = subprocess.run(
                [
                    "pylint",
                    path,
                    "--output-format=json",
                    "--score=y",
                ],
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )

            end = time.time()

            try:
                issues = json.loads(proc.stdout) if proc.stdout.strip() else []
            except json.JSONDecodeError:
                issues = []

            result = {
                "ok": proc.returncode == 0,
                "exit_code": proc.returncode,
                "issues": issues,
                "stderr": proc.stderr or "",
                "wall_time_ms": int((end - start) * 1000),
            }

            return result

        except subprocess.TimeoutExpired as e:
            end = time.time()
            return {
                "ok": False,
                "exit_code": 124,
                "issues": [],
                "stderr": (e.stderr or "") + "\nPYLINT TIMEOUT",
                "wall_time_ms": int((end - start) * 1000),
            }


def run_pylint_analysis(code: str, timeout_s: int = 20) -> Dict[str, Any]:
    import time

    start = time.time()

    with tempfile.TemporaryDirectory() as td:
        path = os.path.join(td, "main.py")
        with open(path, "w", encoding="utf-8") as f:
            f.write(code)

        try:
            proc = subprocess.run(
                [
                    "pylint",
                    path,
                    "--output-format=json",
                    "--score=y",
                ],
                capture_output=True,
                text=True,
                timeout=timeout_s,
            )

            end = time.time()

            try:
                issues = json.loads(proc.stdout) if proc.stdout.strip() else []
            except json.JSONDecodeError:
                issues = []

            summary = {
                "convention": 0,
                "refactor": 0,
                "warning": 0,
                "error": 0,
                "fatal": 0,
            }

            for issue in issues:
                issue_type = issue.get("type", "").lower()
                if issue_type in summary:
                    summary[issue_type] += 1

            total_issues = sum(summary.values())

            score = None
            score_match = re.search(r"rated at\s+([-+]?\d+\.\d+)/10", proc.stderr or "")
            if not score_match:
                score_match = re.search(r"rated at\s+([-+]?\d+\.\d+)/10", proc.stdout or "")

            if score_match:
                try:
                    score = float(score_match.group(1))
                except ValueError:
                    score = None

            return {
                "ok": proc.returncode == 0,
                "exit_code": proc.returncode,
                "score": score,
                "total_issues": total_issues,
                "count_convention": summary["convention"],
                "count_refactor": summary["refactor"],
                "count_warning": summary["warning"],
                "count_error": summary["error"],
                "count_fatal": summary["fatal"],
                "wall_time_ms": int((end - start) * 1000),
            }

        except subprocess.TimeoutExpired as e:
            end = time.time()
            return {
                "ok": False,
                "exit_code": 124,
                "score": None,
                "total_issues": 0,
                "count_convention": 0,
                "count_refactor": 0,
                "count_warning": 0,
                "count_error": 0,
                "count_fatal": 0,
                "wall_time_ms": int((end - start) * 1000),
                "stderr": (e.stderr or "") + "\nPYLINT TIMEOUT",
            }


In [4]:
import sys
writer = make_code_writer()
reviewer = make_code_reviewer()

#task = "Write a function that computes fibonacci recursively till 100."
task1="""
The proper divisors of a number are all the divisors excluding the number itself. For example, the proper divisors of 2828 are 11, 22, 44, 77, and 1414. As the sum of these divisors is equal to 2828, we call it a perfect number.

Interestingly the sum of the proper divisors of 220220 is 284284 and the sum of the proper divisors of 284284 is 220220, forming a chain of two numbers. For this reason, 220220 and 284284 are called an amicable pair.

Perhaps less well known are longer chains. For example, starting with 1249612496, we form a chain of five numbers:
12496→14288→15472→14536→14264(→12496→⋯ )
12496→14288→15472→14536→14264(→12496→⋯)

Since this chain returns to its starting point, it is called an amicable chain.

Find the smallest member of the longest amicable chain with no element exceeding one million.
"""
task="Euler problem 66"
code = generate_python_code(task, writer)
print(code)

run_results=run_python_code(code)
print(run_results)



analysis = run_pylint_analysis(code)
print(analysis)
# sys.exit(0)
review_feedback = reviewer.run(
    f"Here is the code:\n\n{code}\n\nHere are pylint issues:\n\n{json.dumps(analysis, indent=2)}"
)

print(review_feedback.content)

import math

def solve():
    n = int(input())
    if n == 1:
        print("1")
    else:
        print("1")
{'ok': True, 'exit_code': 0, 'stdout': '', 'stderr': '', 'wall_time_ms': 112}
{'ok': False, 'exit_code': 20, 'score': None, 'total_issues': 4, 'count_convention': 3, 'count_refactor': 0, 'count_warning': 1, 'count_error': 0, 'count_fatal': 0, 'wall_time_ms': 1482}
Okay, let's analyze this Python code snippet and provide a review with a score and explanation.

**Analysis:**

The code is extremely simple and functionally correct. It reads input, checks for a single value, and prints "1" if the input is 1, otherwise prints "1".  There are no logical flaws or errors in the code’s core functionality.  However, it's *very* basic and could benefit from improvements in terms of readability and potential for future expansion.

**Score: 4/10**

**Explanation:**

The code is functionally correct and easily understandable. It simply performs a basic input validation and output. However, th